## Image Studio: A tool for image generation, background generation, and image manipulation.

Author: [Bhushan Garware](http://who/bgarware)

##### This software is designed for prototyping (Not for production use) and provided 'as-is', without any express or implied warranty.

In [1]:
import os
import sys
import io
import uuid
import functools
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import vertexai
from vertexai.preview.vision_models import ImageGenerationModel, Image as VertexImage
import gradio as gr
from rembg import remove
from PIL import Image, ImageFont, ImageDraw
import numpy as np

# Load environment variables if python-dotenv is available
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Ensure local temporary and font directories exist
TEMP_DIR = Path("./tmp")
FONTS_DIR = Path("./fonts")
TEMP_DIR.mkdir(parents=True, exist_ok=True)
FONTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Configure Vertex AI Project & Location
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("PROJECT_ID") or "gdc-ai-playground"
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION") or os.environ.get("LOCATION") or "us-central1"

try:
    import google.auth
    os.environ.setdefault("GOOGLE_CLOUD_QUOTA_PROJECT", PROJECT_ID)
    os.environ.setdefault("CLOUDSDK_CORE_PROJECT", PROJECT_ID)
    try:
        credentials, auth_project = google.auth.default(quota_project_id=PROJECT_ID)
        vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials)
    except Exception:
        vertexai.init(project=PROJECT_ID, location=LOCATION)
    print(f"Vertex AI initialized successfully. Project: {PROJECT_ID}, Location: {LOCATION}")
except Exception as e:
    print(f"Notice: Vertex AI initialization encountered an issue: {e}")
    print("If running locally, set GOOGLE_CLOUD_PROJECT in your environment or .env file.")


In [3]:
# ==============================================================================
# MULTI-AGENT COLLABORATIVE SYSTEM ARCHITECTURE
# ==============================================================================
import json
import threading
from dataclasses import dataclass, field
from enum import Enum

DATASTORE_DIR = TEMP_DIR / "vector_datastore"
DATASTORE_DIR.mkdir(parents=True, exist_ok=True)
INDEX_FILE = DATASTORE_DIR / "index.json"

DISCOVERY_ENGINE_DATASTORE_ID = os.environ.get("VERTEX_SEARCH_DATASTORE_ID") or os.environ.get("DISCOVERY_ENGINE_DATASTORE_ID")
DISCOVERY_ENGINE_LOCATION = os.environ.get("DISCOVERY_ENGINE_LOCATION", "global")

class MultimodalVectorDatastore:
    """
    Manages semantic indexing, retrieval, and similarity search for generated images
    using Vertex AI Search (Discovery Engine) & Multimodal Embeddings (multimodalembedding@001).
    """
    def __init__(self):
        self.lock = threading.Lock()
        self.index = self._load_index()
        self.emb_model = None
        self.discovery_client = None
        self._init_embedding_model()
        self._init_discovery_engine()

    def _init_embedding_model(self):
        try:
            from vertexai.vision_models import MultiModalEmbeddingModel
            self.emb_model = MultiModalEmbeddingModel.from_pretrained("multimodalembedding@001")
        except Exception as e:
            print(f"[VectorDatastore] Notice: MultiModalEmbeddingModel init: {e}")

    def _init_discovery_engine(self):
        if PROJECT_ID and DISCOVERY_ENGINE_DATASTORE_ID:
            try:
                from google.cloud import discoveryengine_v1
                self.discovery_client = discoveryengine_v1.SearchServiceClient()
            except Exception as e:
                print(f"[VectorDatastore] Notice: Discovery Engine init: {e}")

    def _load_index(self):
        if INDEX_FILE.exists():
            try:
                with open(INDEX_FILE, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return []
        return []

    def _save_index(self):
        try:
            with open(INDEX_FILE, "w", encoding="utf-8") as f:
                json.dump(self.index, f, indent=2)
        except Exception as e:
            print(f"[VectorDatastore] Error saving index: {e}")

    def compute_text_embedding(self, text: str):
        if not text:
            return None
        if self.emb_model:
            try:
                emb = self.emb_model.get_embeddings(contextual_text=text)
                if emb and emb.text_embedding:
                    return list(emb.text_embedding)
            except Exception:
                pass
        import hashlib
        h = hashlib.sha256(text.lower().strip().encode()).digest()
        vec = [(b / 255.0) * 2 - 1 for b in h] * 4
        norm = np.linalg.norm(vec) + 1e-9
        return [float(x / norm) for x in vec]

    def search_similar(self, query_text: str, similarity_threshold: float = 0.70):
        if self.discovery_client and DISCOVERY_ENGINE_DATASTORE_ID:
            try:
                from google.cloud import discoveryengine_v1
                serving_config = (
                    f"projects/{PROJECT_ID}/locations/{DISCOVERY_ENGINE_LOCATION}/"
                    f"collections/default_collection/dataStores/{DISCOVERY_ENGINE_DATASTORE_ID}/"
                    f"servingConfigs/default_search"
                )
                request = discoveryengine_v1.SearchRequest(
                    serving_config=serving_config,
                    query=query_text,
                    page_size=1,
                )
                response = self.discovery_client.search(request=request)
                for result in response.results:
                    doc = result.document
                    doc_data = dict(doc.struct_data) if hasattr(doc, "struct_data") else {}
                    if "image_path" in doc_data and os.path.exists(doc_data["image_path"]):
                        return {
                            "id": doc.id,
                            "prompt": doc_data.get("prompt", query_text),
                            "image_path": doc_data["image_path"],
                            "engine": "discovery_engine"
                        }, 0.95
            except Exception:
                pass

        with self.lock:
            if not self.index:
                return None, 0.0

            query_vec = self.compute_text_embedding(query_text)
            if not query_vec:
                return None, 0.0

            q_arr = np.array(query_vec)
            best_entry = None
            best_score = -1.0

            for entry in self.index:
                doc_vec = np.array(entry["embedding"])
                if len(doc_vec) != len(q_arr):
                    continue
                score = float(np.dot(q_arr, doc_vec) / (np.linalg.norm(q_arr) * np.linalg.norm(doc_vec) + 1e-9))
                if score > best_score:
                    best_score = score
                    best_entry = entry

            if best_entry and best_score >= similarity_threshold:
                return best_entry, best_score
            return None, max(0.0, best_score)

    def index_image(self, image_pil, prompt: str, entry_id=None):
        if image_pil is None:
            return
        with self.lock:
            try:
                eid = entry_id or f"asset_{uuid.uuid4().hex[:10]}"
                img_path = DATASTORE_DIR / f"{eid}.png"
                image_pil.save(str(img_path), format="PNG")
                
                embedding = self.compute_text_embedding(prompt)
                if embedding:
                    entry = {
                        "id": eid,
                        "prompt": prompt,
                        "image_path": str(img_path),
                        "embedding": embedding,
                        "timestamp": str(uuid.uuid1())
                    }
                    self.index.append(entry)
                    self._save_index()
            except Exception as e:
                print(f"[VectorDatastore] Indexing failed: {e}")

vector_datastore = MultimodalVectorDatastore()


def edit_existing_image(reference_pil, new_prompt: str):
    for model_name in ["gemini-2.5-flash-image", "gemini-3.1-flash-image"]:
        try:
            from vertexai.generative_models import GenerativeModel, Part
            model = GenerativeModel(model_name)

            img_byte_arr = io.BytesIO()
            reference_pil.save(img_byte_arr, format='PNG')
            img_part = Part.from_data(data=img_byte_arr.getvalue(), mime_type="image/png")
            edit_instruction = f"Edit and refine this reference image to match the new prompt: {new_prompt}"

            def fetch_single(idx):
                try:
                    res = model.generate_content(
                        [img_part, edit_instruction],
                        generation_config={"response_modalities": ["TEXT", "IMAGE"]}
                    )
                    if res and res.candidates:
                        for cand in res.candidates:
                            for p in cand.content.parts:
                                if hasattr(p, 'inline_data') and p.inline_data:
                                    return Image.open(io.BytesIO(p.inline_data.data)).resize((1024, 1024), Image.LANCZOS)
                except Exception:
                    pass
                return None

            with ThreadPoolExecutor(max_workers=4) as executor:
                futures = [executor.submit(fetch_single, i) for i in range(4)]
                results = [f.result() for f in futures]
                results = [img for img in results if img is not None]

            if results:
                return results
        except Exception:
            pass
    return []


class AgentDecision(Enum):
    EDIT_EXISTING = "EDIT_EXISTING"
    GENERATE_SCRATCH = "GENERATE_SCRATCH"
    BLOCK_UNSAFE = "BLOCK_UNSAFE"

@dataclass
class AgentExecutionContext:
    raw_prompt: str
    sanitized_prompt: str = ""
    is_prompt_safe: bool = True
    prompt_safety_reason: str = ""
    pii_redacted: bool = False
    decision: AgentDecision = AgentDecision.GENERATE_SCRATCH
    matched_asset_id: str = None
    similarity_score: float = 0.0
    generated_images: list = field(default_factory=list)
    safe_images: list = field(default_factory=list)
    vision_safety_flags: list = field(default_factory=list)
    agent_trace: list = field(default_factory=list)


class CloudArmorPromptGuardAgent:
    INJECTION_PATTERNS = [
        r"(?i)\bignore\s+all\s+(?:previous|prior)\s+instructions\b",
        r"(?i)\bjailbreak\b",
        r"(?i)\bDAN\s+mode\b",
        r"(?i)\bsystem\s+prompt\s+override\b",
    ]

    def inspect(self, ctx: AgentExecutionContext) -> AgentExecutionContext:
        ctx.agent_trace.append("🛡️ **[Cloud Armor Prompt Guard Agent]** Inspecting prompt for injection and exploits...")
        for pattern in self.INJECTION_PATTERNS:
            if re.search(pattern, ctx.raw_prompt):
                ctx.is_prompt_safe = False
                ctx.prompt_safety_reason = f"Security violation detected: '{pattern}'"
                ctx.agent_trace.append(f"❌ **[Cloud Armor] Blocked**: {ctx.prompt_safety_reason}")
                return ctx
        ctx.agent_trace.append("✅ **[Cloud Armor] Verified**: Prompt passed security filters.")
        return ctx


class PIIScrubberAgent:
    def sanitize(self, ctx: AgentExecutionContext) -> AgentExecutionContext:
        ctx.agent_trace.append("🔒 **[Cloud DLP PII Agent]** Scanning prompt for Sensitive Data & PII...")
        clean_text = scrub_pii_dlp(ctx.raw_prompt)
        if clean_text != ctx.raw_prompt:
            ctx.pii_redacted = True
            ctx.agent_trace.append("✂️ **[Cloud DLP] Redacted**: Sensitive PII detected and sanitized.")
        else:
            ctx.agent_trace.append("✅ **[Cloud DLP] Clean**: No sensitive personal data detected.")
        ctx.sanitized_prompt = clean_text
        return ctx


class SearchRetrievalAgent:
    def evaluate(self, ctx: AgentExecutionContext, datastore: MultimodalVectorDatastore) -> AgentExecutionContext:
        ctx.agent_trace.append("🔍 **[Search & Retrieval Agent]** Querying Vertex AI Search / Multimodal Vector Datastore...")
        matched_entry, score = datastore.search_similar(ctx.sanitized_prompt, similarity_threshold=0.70)
        ctx.similarity_score = score
        
        if matched_entry and os.path.exists(matched_entry.get("image_path", "")):
            ctx.decision = AgentDecision.EDIT_EXISTING
            ctx.matched_asset_id = matched_entry["id"]
            pct = score * 100
            ctx.agent_trace.append(f"🎯 **[Search Agent] Match Found**: Asset `{matched_entry['id']}` matched with **{pct:.1f}%** similarity (>= 70%). Routing to **Image Editing Agent**.")
        else:
            ctx.decision = AgentDecision.GENERATE_SCRATCH
            pct = score * 100
            ctx.agent_trace.append(f"🆕 **[Search Agent] No Matching Asset**: Best match **{pct:.1f}%** (<70%). Routing to **Image Generation Agent**.")
        return ctx


class ImageEditingAgent:
    def execute(self, ctx: AgentExecutionContext, datastore: MultimodalVectorDatastore) -> AgentExecutionContext:
        ctx.agent_trace.append(f"🎨 **[Image Editing Agent]** Editing closest candidate asset `{ctx.matched_asset_id}`...")
        for entry in datastore.index:
            if entry["id"] == ctx.matched_asset_id and os.path.exists(entry["image_path"]):
                try:
                    ref_img = Image.open(entry["image_path"])
                    results = edit_existing_image(ref_img, ctx.sanitized_prompt)
                    if results:
                        ctx.generated_images = results
                        ctx.agent_trace.append(f"✅ **[Image Editing Agent]** Successfully refined {len(results)} variations.")
                        return ctx
                except Exception as e:
                    ctx.agent_trace.append(f"⚠️ **[Image Editing Agent]** Error: {e}. Falling back to Generation Agent.")
        ctx.decision = AgentDecision.GENERATE_SCRATCH
        return ctx


class ImageGenerationAgent:
    def execute(self, ctx: AgentExecutionContext) -> AgentExecutionContext:
        ctx.agent_trace.append("✨ **[Image Generation Agent]** Invoking Vertex AI Imagen 4 / Gemini Native synthesis...")
        images = []
        for model_name in ["imagen-4.0-generate-001", "imagen-3.0-generate-002", "gemini-2.5-flash-image"]:
            try:
                from vertexai.preview.vision_models import ImageGenerationModel
                model = ImageGenerationModel.from_pretrained(model_name)
                res = model.generate_images(prompt=ctx.sanitized_prompt, number_of_images=4)
                images = [img._pil_image for img in res.images if hasattr(img, '_pil_image')]
                if images:
                    break
            except Exception:
                continue
        ctx.generated_images = images
        ctx.agent_trace.append(f"✅ **[Image Generation Agent]** Synthesized {len(images)} candidate images from scratch.")
        return ctx


class CloudVisionSafetyAgent:
    def __init__(self):
        self.vision_client = None
        if PROJECT_ID:
            try:
                from google.cloud import vision
                credentials, _ = google.auth.default(quota_project_id=PROJECT_ID)
                self.vision_client = vision.ImageAnnotatorClient(credentials=credentials)
            except Exception:
                pass

    def inspect_and_filter(self, ctx: AgentExecutionContext) -> AgentExecutionContext:
        ctx.agent_trace.append("👁️ **[Cloud Vision Safety Agent]** Moderating generated visual assets with Google Cloud Vision SafeSearch Detection...")
        safe_list = []
        for idx, img in enumerate(ctx.generated_images):
            if img is None:
                continue
            is_safe = True
            flag_reason = ""
            if self.vision_client:
                try:
                    from google.cloud import vision
                    buf = io.BytesIO()
                    img.save(buf, format="PNG")
                    v_img = vision.Image(content=buf.getvalue())
                    response = self.vision_client.safe_search_detection(image=v_img)
                    safe = response.safe_search_annotation
                    if safe.adult in (vision.Likelihood.LIKELY, vision.Likelihood.VERY_LIKELY):
                        is_safe = False
                        flag_reason = "Adult content detected"
                    elif safe.violence in (vision.Likelihood.LIKELY, vision.Likelihood.VERY_LIKELY):
                        is_safe = False
                        flag_reason = "Violence detected"
                except Exception:
                    pass

            if is_safe:
                safe_list.append(img)
            else:
                ctx.agent_trace.append(f"🚫 **[Cloud Vision] Redacted**: Image #{idx+1} blocked ({flag_reason}).")
                blocked_canvas = Image.new("RGB", (1024, 1024), color=(30, 30, 30))
                draw = ImageDraw.Draw(blocked_canvas)
                draw.text((200, 500), f"[CONTENT BLOCKED: {flag_reason}]", fill=(255, 100, 100))
                safe_list.append(blocked_canvas)

        ctx.safe_images = safe_list
        ctx.agent_trace.append(f"✅ **[Cloud Vision Safety Agent]** Verified {len(safe_list)} images compliant with safety policy.")
        return ctx


class IndexingMemoryAgent:
    def record(self, ctx: AgentExecutionContext, datastore: MultimodalVectorDatastore) -> AgentExecutionContext:
        ctx.agent_trace.append("💾 **[Indexing & Memory Agent]** Indexing verified visual assets into Vertex AI Vector Datastore...")
        count = 0
        for img in ctx.safe_images:
            if img:
                datastore.index_image(img, ctx.sanitized_prompt)
                count += 1
        ctx.agent_trace.append(f"✅ **[Indexing Agent]** Indexed {count} assets for future retrieval.")
        return ctx


class ImageSenseMultiAgentOrchestrator:
    def __init__(self, datastore: MultimodalVectorDatastore):
        self.datastore = datastore
        self.guard_agent = CloudArmorPromptGuardAgent()
        self.pii_agent = PIIScrubberAgent()
        self.search_agent = SearchRetrievalAgent()
        self.edit_agent = ImageEditingAgent()
        self.gen_agent = ImageGenerationAgent()
        self.safety_agent = CloudVisionSafetyAgent()
        self.memory_agent = IndexingMemoryAgent()

    def process(self, raw_prompt: str):
        ctx = AgentExecutionContext(raw_prompt=raw_prompt)
        ctx = self.guard_agent.inspect(ctx)
        if not ctx.is_prompt_safe:
            return [None, None, None, None], "### 🤖 Multi-Agent Execution Trace\n\n" + "\n\n".join(ctx.agent_trace)
        ctx = self.pii_agent.sanitize(ctx)
        ctx = self.search_agent.evaluate(ctx, self.datastore)
        if ctx.decision == AgentDecision.EDIT_EXISTING:
            ctx = self.edit_agent.execute(ctx, self.datastore)
        if ctx.decision == AgentDecision.GENERATE_SCRATCH or not ctx.generated_images:
            ctx = self.gen_agent.execute(ctx)
        ctx = self.safety_agent.inspect_and_filter(ctx)
        ctx = self.memory_agent.record(ctx, self.datastore)

        images = ctx.safe_images[:]
        while len(images) < 4:
            images.append(None)
        trace_md = "### 🤖 Multi-Agent Collaborative Execution Trace\n\n" + "\n\n".join(ctx.agent_trace)
        return images[:4], trace_md

multi_agent_orchestrator = ImageSenseMultiAgentOrchestrator(vector_datastore)


def image_generation_completion(input):
    if not input or not str(input).strip():
        gr.Warning("Please enter an image prompt first.")
        return [None, None, None, None, "⚠️ Please enter an image prompt first."]
    images, trace_md = multi_agent_orchestrator.process(str(input).strip())
    return [images[0], images[1], images[2], images[3], trace_md]

In [4]:
import re

def scrub_pii_dlp(text: str) -> str:
    """
    Automated PII Scrubbing via Cloud DLP:
    Detects, masks, and redacts Sensitive Data / PII (names, emails, phone numbers,
    addresses, credit cards, SSNs) from user prompts and metadata prior to LLM processing.
    Falls back to deterministic regex de-identification if Cloud DLP API is unreachable.
    """
    if not text or not isinstance(text, str):
        return text

    sanitized_text = text

    # 1. Try Google Cloud DLP API
    if PROJECT_ID:
        try:
            from google.cloud import dlp_v2
            dlp_client = dlp_v2.DlpServiceClient()
            parent = f"projects/{PROJECT_ID}/locations/global"

            info_types = [
                {"name": "EMAIL_ADDRESS"},
                {"name": "PHONE_NUMBER"},
                {"name": "PERSON_NAME"},
                {"name": "CREDIT_CARD_NUMBER"},
                {"name": "STREET_ADDRESS"},
                {"name": "US_SOCIAL_SECURITY_NUMBER"},
                {"name": "IP_ADDRESS"},
            ]
            inspect_config = {
                "info_types": info_types,
                "min_likelihood": dlp_v2.Likelihood.POSSIBLE,
                "include_quote": True,
            }

            deidentify_config = {
                "info_type_transformations": {
                    "transformations": [
                        {
                            "primitive_transformation": {
                                "replace_with_info_type_config": {}
                            }
                        }
                    ]
                }
            }

            item = {"value": text}
            response = dlp_client.deidentify_content(
                request={
                    "parent": parent,
                    "deidentify_config": deidentify_config,
                    "inspect_config": inspect_config,
                    "item": item,
                }
            )
            if response and response.item and response.item.value:
                sanitized_text = response.item.value
                if sanitized_text != text:
                    print(f"[Cloud DLP] Sensitive PII detected and redacted: '{text}' -> '{sanitized_text}'")
        except Exception:
            pass

    # 2. Defense-in-Depth Regex PII Masking
    sanitized_text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[EMAIL_ADDRESS]', sanitized_text)
    sanitized_text = re.sub(r'\b(?:\d[ -]*?){13,19}\b', '[CREDIT_CARD_NUMBER]', sanitized_text)
    sanitized_text = re.sub(r'(\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', '[PHONE_NUMBER]', sanitized_text)
    sanitized_text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[US_SOCIAL_SECURITY_NUMBER]', sanitized_text)

    return sanitized_text


def prompt_generation(persona, signal, theme, lighting, quality, extra_desc, TextOnImg, TextFmtOnImg):
    """
    Generates an enriched image generation prompt using Gemini / Vertex AI text models,
    with a robust fallback to structured template composition and automated Cloud DLP PII scrubbing.
    """
    # Scrub PII from all user prompt inputs prior to LLM processing
    persona = scrub_pii_dlp(persona)
    signal = scrub_pii_dlp(signal)
    theme = scrub_pii_dlp(theme)
    extra_desc = scrub_pii_dlp(extra_desc)
    TextOnImg = scrub_pii_dlp(TextOnImg)

    params_list = [p for p in [persona, signal, theme, lighting, quality, extra_desc] if p and str(p).strip()]
    params_list_str = ", ".join(params_list) if params_list else "A high quality product showcase"
    
    few_shot_prompt = f"""You are an expert in writing prompts for Image Generation Models. Using the provided phrases and keywords, concatenate them and add on some realistic details to generate a logical and meaningful prompt that can be used for image generation.

input: Young woman, wearing NIKE sneakers, tennis court, Natural, HD image, Photo.
output: A Photo of Young woman wearing NIKE sneakers on tennis court, Natural lighting, HD quality photo clicked by a professional photographer.
input: Old man, wearing sports shoes, vegetable market, warm, high-quality, sketch.
output: A sketch of old man wearing sports shoes in vegetable market, warm lighting, high quality sketch drawn by a professional painter.
input: {params_list_str}
output:"""

    output_prompt = ""
    # Try active Gemini text models on Vertex AI
    for model_name in ["gemini-2.5-flash-image", "gemini-2.0-flash", "gemini-1.5-flash", "gemini-1.5-pro"]:
        try:
            from vertexai.generative_models import GenerativeModel
            model = GenerativeModel(model_name)
            response = model.generate_content(few_shot_prompt)
            if response and response.text:
                output_prompt = response.text.strip()
                break
        except Exception:
            continue

    # Fallback to local deterministic template composition if LLM is unavailable
    if not output_prompt:
        subject_part = persona or "subject"
        action_part = f", {signal}" if signal else ""
        theme_part = f" in {theme}" if theme else ""
        light_part = f", {lighting} lighting" if lighting else ""
        qual_part = f", {quality}" if quality else ""
        type_part = f"A {extra_desc} of" if extra_desc else "A photo of"
        output_prompt = f"{type_part} {subject_part}{action_part}{theme_part}{light_part}{qual_part}."

    # Add text overlay instructions if specified
    if TextOnImg and str(TextOnImg).strip():
        txt = str(TextOnImg).strip()
        if TextFmtOnImg and str(TextFmtOnImg).strip():
            output_prompt += f" Add a title to the corner that reads '{txt}' in {str(TextFmtOnImg).strip()}."
        else:
            output_prompt += f" Add a title to the corner that reads '{txt}'."
            
    return scrub_pii_dlp(output_prompt)

In [5]:
def background_generation(input_image, prompt):
    """
    Generates three background variations for a given input image based on a text prompt.
    Supports Vertex AI Imagen capability models with seamless fallback to Gemini Multimodal image editing.
    """
    if input_image is None:
        gr.Warning("Please upload an input image.")
        return [None, None, None]
        
    if not prompt or not str(prompt).strip():
        gr.Warning("Please provide a background prompt.")
        return [None, None, None]

    prompt_text = str(prompt).strip()
    req_id = uuid.uuid4().hex[:8]
    base_img_path = TEMP_DIR / f"base_img_{req_id}.png"
    created_temp_files = [base_img_path]

    try:
        input_image.save(str(base_img_path))
        
        # 1. Try Imagen editing models
        try:
            from vertexai.preview.vision_models import Image as VertexImage, ImageGenerationModel
            model = None
            for model_name in ["imagen-4.0-generate-001", "imagen-3.0-capability-001", "imagen-3.0-generate-002"]:
                try:
                    model = ImageGenerationModel.from_pretrained(model_name)
                    break
                except Exception:
                    continue

            if model is not None:
                base_img = VertexImage.load_from_file(location=str(base_img_path))
                images = model.edit_image(
                    base_image=base_img,
                    prompt=prompt_text,
                    number_of_images=3,
                    edit_mode="product-image"
                )

                results = []
                for i, img in enumerate(images):
                    save_path = TEMP_DIR / f"im_{req_id}_{i}.png"
                    created_temp_files.append(save_path)
                    img.save(location=str(save_path), include_generation_parameters=True)
                    opened = Image.open(str(save_path)).resize((1024, 1024), Image.LANCZOS)
                    results.append(opened)

                if results:
                    while len(results) < 3:
                        results.append(None)
                    return results
        except Exception as imagen_err:
            print(f"[ImageStudio Notice] Imagen background editing unavailable ({imagen_err}), switching to Gemini multimodal...")

        # 2. Fallback: Gemini Native Multimodal background synthesis (3 in parallel)
        for model_name in ["gemini-2.5-flash-image", "gemini-3.1-flash-image", "gemini-3-pro-image-preview"]:
            try:
                from vertexai.generative_models import GenerativeModel, Part
                model = GenerativeModel(model_name)

                # Convert input PIL image to bytes part for multimodal query
                img_byte_arr = io.BytesIO()
                input_image.save(img_byte_arr, format='PNG')
                img_part = Part.from_data(data=img_byte_arr.getvalue(), mime_type="image/png")
                edit_instruction = f"Place this product into a new background scene: {prompt_text}. Maintain product fidelity and lighting coherence."

                def fetch_bg_variation(idx):
                    try:
                        res = model.generate_content(
                            [img_part, edit_instruction],
                            generation_config={"response_modalities": ["TEXT", "IMAGE"]}
                        )
                        if res and res.candidates:
                            for cand in res.candidates:
                                for p in cand.content.parts:
                                    if hasattr(p, 'inline_data') and p.inline_data:
                                        return Image.open(io.BytesIO(p.inline_data.data)).resize((1024, 1024), Image.LANCZOS)
                    except Exception as ex:
                        print(f"[ImageStudio Background {model_name} #{idx} Error] {ex}")
                    return None

                with ThreadPoolExecutor(max_workers=3) as executor:
                    futures = [executor.submit(fetch_bg_variation, i) for i in range(3)]
                    results = [f.result() for f in futures]
                    results = [img for img in results if img is not None]

                if results:
                    while len(results) < 3:
                        results.append(None)
                    return results
            except Exception as gemini_err:
                print(f"[ImageStudio Error] Gemini background variation failed on {model_name}: {gemini_err}")

        gr.Warning("Background generation failed across candidate models. Verify Vertex AI project & billing.")
        return [None, None, None]
    except Exception as e:
        error_msg = f"Background generation failed: {e}"
        print(f"[ImageStudio Error] {error_msg}")
        gr.Warning(f"{error_msg}. Verify Vertex AI permissions.")
        return [None, None, None]
    finally:
        # Clean up temporary disk buffers
        for temp_file in created_temp_files:
            try:
                if temp_file.exists():
                    temp_file.unlink()
            except Exception:
                pass


def insert_image(im1, im2, angle, height, width, left, top):
    """
    Inserts a product/foreground image (im2) into a base background image (im1)
    after resizing, rotating, and cleanly removing the background using rembg.
    """
    if im1 is None:
        gr.Warning("Please upload a Background Image.")
        return None
    if im2 is None:
        gr.Warning("Please upload a Product Image to insert.")
        return im1
        
    try:
        base_img = im1.copy().convert("RGBA")
        
        # Remove background of product image to get RGBA with transparent alpha
        fg_rgba = remove(im2)
        if fg_rgba.mode != "RGBA":
            fg_rgba = fg_rgba.convert("RGBA")
        
        # Resize foreground image
        target_w = max(10, int(width))
        target_h = max(10, int(height))
        fg_resized = fg_rgba.resize((target_w, target_h), Image.LANCZOS)
        
        # Rotate foreground image (counter-clockwise)
        fg_rotated = fg_resized.rotate(-float(angle), resample=Image.BICUBIC, expand=True)
        
        # Create transparent overlay layer matching base image size
        overlay_layer = Image.new("RGBA", base_img.size, (0, 0, 0, 0))
        overlay_layer.paste(fg_rotated, (int(left), int(top)), mask=fg_rotated)
        
        # Alpha composite base and overlay
        result = Image.alpha_composite(base_img, overlay_layer)
        return result.convert("RGB")
    except Exception as e:
        gr.Warning(f"Error inserting image: {e}")
        return im1


def insert_more_images(input_image):
    """
    Allows chaining insertions: moves the output image back into the background slot.
    """
    return [input_image, None, None]


def AddLogo(MainImage, LogoImage, factor, opacity, left, top):
    """
    Adds a logo overlay onto a main image with scaling, opacity, and positioning.
    """
    if MainImage is None:
        gr.Warning("Please upload a Background Image.")
        return None
    if LogoImage is None:
        gr.Warning("Please upload a Logo Image.")
        return MainImage

    try:
        # Load main image
        if isinstance(MainImage, str):
            main_pil = Image.open(MainImage).convert("RGBA")
        elif isinstance(MainImage, np.ndarray):
            main_pil = Image.fromarray(MainImage).convert("RGBA")
        else:
            main_pil = MainImage.copy().convert("RGBA")

        # Load logo image
        if isinstance(LogoImage, str):
            logo_pil = Image.open(LogoImage).convert("RGBA")
        elif isinstance(LogoImage, np.ndarray):
            logo_pil = Image.fromarray(LogoImage).convert("RGBA")
        else:
            logo_pil = LogoImage.copy().convert("RGBA")

        # Resize logo by factor
        orig_w, orig_h = logo_pil.size
        new_w = max(5, int(orig_w * float(factor)))
        new_h = max(5, int(orig_h * float(factor)))
        logo_resized = logo_pil.resize((new_w, new_h), Image.LANCZOS)

        # Adjust opacity
        alpha_scale = max(0.0, min(1.0, float(opacity) / 100.0))
        r, g, b, a = logo_resized.split()
        a = a.point(lambda p: int(p * alpha_scale))
        logo_resized = Image.merge("RGBA", (r, g, b, a))

        # Composite onto canvas
        overlay_layer = Image.new("RGBA", main_pil.size, (0, 0, 0, 0))
        overlay_layer.paste(logo_resized, (int(left), int(top)), mask=logo_resized)
        
        result = Image.alpha_composite(main_pil, overlay_layer)
        return result.convert("RGB")
    except Exception as e:
        gr.Warning(f"Error adding logo: {e}")
        return MainImage


@functools.lru_cache(maxsize=128)
def find_font(font_name, font_size):
    """
    Resolves font by checking fonts directory, temp directory, system font directories,
    and falls back safely to default font to avoid OSError.
    """
    font_size = max(8, int(font_size))
    candidate_paths = [
        FONTS_DIR / f"{font_name}.ttf",
        FONTS_DIR / f"{font_name}.otf",
        TEMP_DIR / f"{font_name}.ttf",
        TEMP_DIR / f"{font_name}.otf",
        Path(f"/System/Library/Fonts/Supplemental/{font_name}.ttf"),
        Path(f"/System/Library/Fonts/{font_name}.ttf"),
        Path(f"/Library/Fonts/{font_name}.ttf"),
        Path(f"/usr/share/fonts/truetype/{font_name}.ttf"),
        Path(f"C:/Windows/Fonts/{font_name}.ttf"),
    ]
    for p in candidate_paths:
        if p.exists():
            try:
                return ImageFont.truetype(str(p), font_size)
            except Exception:
                pass

    try:
        return ImageFont.truetype(font_name, font_size)
    except Exception:
        pass

    try:
        return ImageFont.load_default(size=font_size)
    except TypeError:
        return ImageFont.load_default()


def AddText(bkg_image, input_text, font, font_size, R, G, B, left, top):
    """
    Renders custom styled text onto an image.
    """
    if bkg_image is None:
        gr.Warning("Please upload a Background Image.")
        return None
    if not input_text:
        return bkg_image

    try:
        img = bkg_image.copy().convert("RGB")
        draw = ImageDraw.Draw(img)
        loaded_font = find_font(font, int(font_size))
        color = (int(R), int(G), int(B))
        draw.text((int(left), int(top)), str(input_text), fill=color, font=loaded_font)
        return img
    except Exception as e:
        gr.Warning(f"Error adding text: {e}")
        return bkg_image


def AddMoreText(input_image):
    """
    Allows chaining text additions: moves the output image back into the background slot.
    """
    return [input_image, input_image, ""]

In [6]:
available_fonts = ["Arial", "Arial Black", "Arial Bold", "Arial Italic", "Arial Narrow", "Arial Rounded Bold"]
if FONTS_DIR.exists():
    for f in FONTS_DIR.glob("*.ttf"):
        if f.stem not in available_fonts:
            available_fonts.append(f.stem)
for extra in ["ClarendonBT", "FUTURAM", "SerpentineBoldItalic", "Helvetica", "Courier New", "Times New Roman"]:
    if extra not in available_fonts:
        available_fonts.append(extra)

gr.close_all()
with gr.Blocks(theme=gr.themes.Soft(), title="Image Studio") as demo:
    with gr.Tab("Image Generation"):
        # Prompt Generation Part
        with gr.Row():
            with gr.Column(scale=1):
                Persona = gr.Textbox(label="Subject", info="e.g. Old woman, Man in 60s, Winter Boots")
            with gr.Column(scale=1):
                Signals = gr.Textbox(label="Action", info="e.g. Standing, kept on the table")
            with gr.Column(scale=1):
                Theme = gr.Textbox(label="Theme", info="e.g. On tennis court, in the market")
        with gr.Row():
            with gr.Column(scale=1):
                photo_modifiers = gr.Dropdown(["Dramatic", "Natural", "Warm", "Cold", "Cinematic"], label="Photography Modifiers", value="Natural")
            with gr.Column(scale=1):
                quality_modifiers = gr.Dropdown(["high-quality", "beautiful", "stylized", "4K", "HDR", "By a professional photographer"], label="Image Quality Modifier", value="By a professional photographer")
            with gr.Column(scale=1):
                other_desc = gr.Dropdown(["Photo", "painting", "Sketch", "Digital Art"], label="Image Type", value="Photo")

        with gr.Row():
            with gr.Column(scale=1):
                TextOnImg = gr.Textbox(label="Optional, Text on Image", info="e.g. 30% OFF on Flight Bookings")
            with gr.Column(scale=1):
                TextFmtOnImg = gr.Textbox(label="Optional, Text format", info="e.g. Pink and white block letters")      
            
        with gr.Row():
            btn = gr.Button("Generate Prompt", variant="secondary")    
        
        # Image Generation part with Vertex AI Vector Search
        search_status = gr.Markdown("🟢 **Vertex AI Vector Search Ready**: Incoming prompts will search existing image datastore (`multimodalembedding@001`). If a similar asset exists (≥70%), it will be edited directly; otherwise, new images will be synthesized from scratch and indexed.")

        with gr.Row():
            with gr.Column(scale=1):
                image_prompt = gr.Textbox(label="Image Generation Prompt", lines=3)
                
        btn.click(fn=prompt_generation, inputs=[Persona, Signals, Theme, photo_modifiers, quality_modifiers, other_desc, TextOnImg, TextFmtOnImg], outputs=image_prompt)

        with gr.Row():
            with gr.Column(scale=1):    
                img_btn = gr.Button("Generate 4 Images (Vector Search & Synthesize)", variant="primary")

        with gr.Row():
            with gr.Column():
                output_image_1 = gr.Image(label="Result Image 1", type="pil")
            with gr.Column():
                output_image_2 = gr.Image(label="Result Image 2", type="pil")
        with gr.Row():
            with gr.Column():
                output_image_3 = gr.Image(label="Result Image 3", type="pil")
            with gr.Column():
                output_image_4 = gr.Image(label="Result Image 4", type="pil")

        components = [image_prompt, output_image_1, output_image_2, output_image_3, output_image_4, search_status]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(components)
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")

        img_btn.click(fn=image_generation_completion, inputs=[image_prompt], outputs=[output_image_1, output_image_2, output_image_3, output_image_4, search_status])
        
    with gr.Tab("Background Generation"):
        with gr.Row():
            with gr.Column():
                b_input_image = gr.Image(label="Input Image", type="pil")
            with gr.Column():
                b_text_prompt = gr.Textbox(label="Background Prompt", info="e.g. Blue sea with white sand", lines=3)
                b_btn = gr.Button("Change Background", variant="primary")
        with gr.Row():
            with gr.Column():
                b_output_image1 = gr.Image(label="Result Image 1", type="pil")
            with gr.Column():
                b_output_image2 = gr.Image(label="Result Image 2", type="pil")
            with gr.Column():
                b_output_image3 = gr.Image(label="Result Image 3", type="pil")
                    
        b_components = [b_input_image, b_text_prompt, b_output_image1, b_output_image2, b_output_image3]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(b_components)
                    
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")
                
        b_btn.click(fn=background_generation, inputs=[b_input_image, b_text_prompt], outputs=[b_output_image1, b_output_image2, b_output_image3])
    
    with gr.Tab("Insert Image"):
        with gr.Row():
            with gr.Column():
                i_bkg_image = gr.Image(label="Background Image", type="pil")
            with gr.Column():
                i_prd_image = gr.Image(label="Product Image", type="pil")
        with gr.Row():
            with gr.Column():
                i_angle = gr.Slider(-180, 180, value=0, label="Angle", info="Angle of Rotation")
                i_height = gr.Slider(10, 2000, value=256, label="Height", info="Product Height")
                i_width = gr.Slider(10, 2000, value=256, label="Width", info="Product Width")
                i_left = gr.Slider(0, 3000, value=100, label="Towards Right", info="Ref top left corner")
                i_down = gr.Slider(0, 3000, value=100, label="Towards Down", info="Ref top left corner")
                i_btn = gr.Button("Insert Image", variant="primary")
            with gr.Column():
                i_output_image = gr.Image(label="Result Image", type="pil")
                ii_btn = gr.Button("Insert Another Image", variant="secondary")
                    
        i_components = [i_bkg_image, i_prd_image, i_output_image]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(i_components)
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")
                
        i_btn.click(fn=insert_image, inputs=[i_bkg_image, i_prd_image, i_angle, i_height, i_width, i_left, i_down], outputs=i_output_image)
        ii_btn.click(fn=insert_more_images, inputs=i_output_image, outputs=[i_bkg_image, i_prd_image, i_output_image])
    
    with gr.Tab("Insert Logo"):
        with gr.Row():
            with gr.Column():
                l_bkg_image = gr.Image(label="Background Image", type="pil")
            with gr.Column():
                l_prd_image = gr.Image(label="Logo Image", type="pil")
        with gr.Row():
            with gr.Column():
                l_factor = gr.Slider(0.1, 5, value=1, label="Factor", info="Scaling Factor")
                l_opacity = gr.Slider(0, 100, value=80, label="Opacity (%)", info="Opacity")
                l_left = gr.Slider(0, 3000, value=100, label="Towards Right", info="Ref top left corner")
                l_down = gr.Slider(0, 3000, value=100, label="Towards Down", info="Ref top left corner")
                l_btn = gr.Button("Insert Logo", variant="primary")
            with gr.Column():
                l_output_image = gr.Image(label="Result Image", type="pil")
                    
        l_components = [l_bkg_image, l_prd_image, l_output_image]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(l_components)
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")
                
        l_btn.click(fn=AddLogo, inputs=[l_bkg_image, l_prd_image, l_factor, l_opacity, l_left, l_down], outputs=l_output_image)
    
    with gr.Tab("Insert Text"):
        with gr.Row():
            with gr.Column():
                t_bkg_image = gr.Image(label="Background Image", type="pil")
                    
            with gr.Column():
                t_text = gr.Textbox(label="Text", info="Enter Sample Text", value="SAMPLE TEXT")
                t_font = gr.Dropdown(available_fonts, label="Font", info="Select the font", value="Arial", allow_custom_value=True)
                t_size = gr.Slider(10, 200, value=36, label="Font Size", info="Select Font Size")
                t_R = gr.Slider(0, 255, value=255, label="R Component", info="R Color Component Intensity")
                t_G = gr.Slider(0, 255, value=255, label="G Component", info="G Color Component Intensity")
                t_B = gr.Slider(0, 255, value=255, label="B Component", info="B Color Component Intensity")
                t_left = gr.Slider(0, 3000, value=100, label="Towards Right", info="Ref top left corner")
                t_down = gr.Slider(0, 3000, value=100, label="Towards Down", info="Ref top left corner")
                t_btn = gr.Button("Insert Text", variant="primary")
                    
            with gr.Column():
                t_output_image = gr.Image(label="Result Image", type="pil")
                tt_btn = gr.Button("Insert More Text", variant="secondary")
                    
        t_components = [t_bkg_image, t_text, t_output_image]
        with gr.Row():
            with gr.Column(scale=1):
                gr.ClearButton(t_components)
        with gr.Row():
            gr.Markdown("Author: [Bhushan Garware](http://who/bgarware)")
                
        t_btn.click(fn=AddText, inputs=[t_bkg_image, t_text, t_font, t_size, t_R, t_G, t_B, t_left, t_down], outputs=t_output_image)
        tt_btn.click(fn=AddMoreText, inputs=t_output_image, outputs=[t_bkg_image, t_output_image, t_text])

# Launch configuration for local environment on port 8080
port = int(os.environ.get("PORT", 8080))
demo.queue().launch(server_name="0.0.0.0", server_port=port, share=False)

Running on local URL:  http://127.0.0.1:7861
IMPORTANT: You are using gradio version 4.29.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://64b5214d4ce6108be7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
